# Experiment 9: compact features plus leakage-safe target features

This experiment tests the combined feature set built on `compact`:

1. 24-month global and town/flat-type target lags;
2. a town/flat-type historical target mean; and
3. the nearest MRT station name;
4. the nearest tourist-attraction name; and
5. the nearest national-monument name.

The test set has 17 distinct approval months (2025-03 to 2026-07). This experiment intentionally uses `LAG_MONTHS = 24` to test a two-year target lag. Target-derived values for validation and test data are fitted from earlier labelled training rows only. The training target-mean feature is constructed from prior months only, so it does not include the row's own target or its current month's targets.

In [53]:
from pathlib import Path
import json
import os
import sys

import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from scipy.spatial import cKDTree
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

configured_root = os.environ.get('KDDM_PROJECT_ROOT')
candidate_roots = ([Path(configured_root)] if configured_root else []) + [
    Path.cwd(), *Path.cwd().parents, Path('/content/KDDM-project'),
    Path('/content/drive/MyDrive/KDDM-project'),
]
project_root = next((p for p in candidate_roots if (p / 'exploration/metul/preprocessing.py').exists()), None)
if project_root is None:
    raise FileNotFoundError('Set KDDM_PROJECT_ROOT to the project directory.')
experiment_dir = project_root / 'exploration' / 'metul'
sys.path.insert(0, str(experiment_dir))

from preprocessing import RentalPreprocessor, load_datasets

TARGET = 'MONTHLY_RENT'
GROUP_COLUMNS = ['TOWN', 'FLAT_TYPE']
TIME_LIMIT_SECONDS = int(os.environ.get('AUTOGLUON_TIME_LIMIT', '600'))
RUN_ID = os.environ.get('AUTOGLUON_RUN_ID', pd.Timestamp.now().strftime('%Y%m%d_%H%M%S'))
MODEL_HYPERPARAMETERS = {'GBM': {}, 'CAT': {}}

compact_features = [
    'TOWN', 'BLOCK', 'STREET', 'FLAT_TYPE', 'FLAT_MODEL',
    'FLOOR_AREA_SQM', 'approval_date_num', 'approval_month',
    'coe_quota_premium_mean', 'YEAR_COMPLETED', 'MAX_FLOOR',
    'mrt_dist_nearest_km', 'mall_dist_nearest_km',
]


In [54]:
data_dir, datasets = load_datasets(project_root / 'data')
raw_train = datasets['train'].copy()
raw_test = datasets['test'].copy()

raw_train['period'] = pd.to_datetime(raw_train['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
raw_test['period'] = pd.to_datetime(raw_test['RENT_APPROVAL_DATE'] + '-01').dt.to_period('M')
test_periods = np.sort(raw_test['period'].unique())
TEST_HORIZON_MONTHS = len(test_periods)
assert TEST_HORIZON_MONTHS == 17, f'Expected 17 test months, found {TEST_HORIZON_MONTHS}'
LAG_MONTHS = 24
print(f'Test horizon: {TEST_HORIZON_MONTHS} months ({test_periods[0]} to {test_periods[-1]}); target lag: {LAG_MONTHS} months')

periods = np.sort(raw_train['period'].unique())
validation_cutoff = periods[int(len(periods) * 0.80)]
raw_fit = raw_train.loc[raw_train['period'] < validation_cutoff].copy()
raw_valid = raw_train.loc[raw_train['period'] >= validation_cutoff].copy()
print(f'Validation cutoff: {validation_cutoff}; fit rows: {len(raw_fit):,}; validation rows: {len(raw_valid):,}')


Test horizon: 17 months (2025-03 to 2026-07); target lag: 24 months
Validation cutoff: 2024-05; fit rows: 117,988; validation rows: 32,012


In [55]:
class TargetFeatureTransformer:
    """Create target features from labelled history only."""

    def __init__(self, lag_months, group_columns=GROUP_COLUMNS, target=TARGET):
        self.lag_months = lag_months
        self.group_columns = group_columns
        self.target = target

    @staticmethod
    def _keys(frame):
        return frame[GROUP_COLUMNS + ['period']].copy()

    def fit(self, labelled_frame):
        required = set(self.group_columns + ['period', self.target])
        missing = required - set(labelled_frame.columns)
        if missing:
            raise ValueError(f'Missing required columns: {sorted(missing)}')

        history = labelled_frame[self.group_columns + ['period', self.target]].copy()
        self.global_mean_ = history[self.target].mean()
        self.group_mean_ = history.groupby(self.group_columns)[self.target].mean()
        self.monthly_target_mean_ = history.groupby('period')[self.target].mean()
        self.group_period_mean_ = history.groupby(
            self.group_columns + ['period'], as_index=False
        )[self.target].mean()
        self.lag_lookup_ = self.group_period_mean_.rename(columns={
            'period': 'source_period', self.target: 'town_flat_type_target_lag'
        })
        return self

    def transform(self, frame):
        """Map features using only the labelled history supplied to fit()."""
        result = self._keys(frame)
        result['monthly_target_lag'] = (
            result['period'] - self.lag_months
        ).map(self.monthly_target_mean_).fillna(self.global_mean_)
        result['source_period'] = result['period'] - self.lag_months
        result = result.merge(
            self.lag_lookup_, on=self.group_columns + ['source_period'], how='left'
        ).drop(columns='source_period')
        group_mean = pd.Series(
            result.set_index(self.group_columns).index.map(self.group_mean_), index=result.index
        )
        result['town_flat_type_target_mean'] = group_mean.fillna(self.global_mean_)
        result['town_flat_type_target_lag'] = result['town_flat_type_target_lag'].fillna(
            result['town_flat_type_target_mean']
        )
        return result[[
            'monthly_target_lag', 'town_flat_type_target_lag', 'town_flat_type_target_mean'
        ]]

    def fit_transform_train(self, labelled_frame):
        """Return training features whose target means use earlier months only."""
        self.fit(labelled_frame)
        history = labelled_frame[self.group_columns + ['period', self.target]].copy()
        monthly = (
            history.groupby(self.group_columns + ['period'], as_index=False)[self.target].agg(['sum', 'count']).reset_index()
        )
        monthly = monthly.sort_values(self.group_columns + ['period'])
        monthly['prior_sum'] = monthly.groupby(self.group_columns)['sum'].cumsum() - monthly['sum']
        monthly['prior_count'] = monthly.groupby(self.group_columns)['count'].cumsum() - monthly['count']
        monthly['town_flat_type_target_mean'] = monthly['prior_sum'] / monthly['prior_count'].replace(0, np.nan)

        global_monthly = history.groupby('period')[self.target].agg(['sum', 'count']).sort_index()
        global_monthly['prior_sum'] = global_monthly['sum'].cumsum() - global_monthly['sum']
        global_monthly['prior_count'] = global_monthly['count'].cumsum() - global_monthly['count']
        global_prior_mean = global_monthly['prior_sum'] / global_monthly['prior_count'].replace(0, np.nan)

        result = self._keys(labelled_frame)
        result = result.merge(
            monthly[self.group_columns + ['period', 'town_flat_type_target_mean']],
            on=self.group_columns + ['period'], how='left'
        )
        result['town_flat_type_target_mean'] = result['town_flat_type_target_mean'].fillna(
            result['period'].map(global_prior_mean)
        )
        # For training rows, use the same town/flat-type mean from exactly
        # LAG_MONTHS earlier. If unavailable, use the prior-only group mean.
        result['source_period'] = result['period'] - self.lag_months
        result = result.merge(
            self.lag_lookup_, on=self.group_columns + ['source_period'], how='left'
        ).drop(columns='source_period')
        result['town_flat_type_target_lag'] = result['town_flat_type_target_lag'].fillna(
            result['town_flat_type_target_mean']
        )
        # The global lag is the all-listings monthly mean at the same offset.
        # Its training fallback is also strictly prior-only.
        result['monthly_target_lag'] = (result['period'] - self.lag_months).map(
            self.monthly_target_mean_
        ).fillna(result['period'].map(global_prior_mean))
        return result[[
            'monthly_target_lag', 'town_flat_type_target_lag', 'town_flat_type_target_mean'
        ]]


## Leakage safeguards

- The validation target features are fitted on `raw_fit` only; no validation targets are read.
- The test target features are fitted on the full labelled training set only; no test targets are read.
- The training town/flat-type mean uses targets from **strictly earlier months**.
- `monthly_target_lag` is the overall mean `MONTHLY_RENT` from exactly 24 months earlier; unavailable values fall back to the fitted global mean. `town_flat_type_target_lag` is the same-town/flat-type mean from 24 months earlier; unavailable values fall back to its fitted town/flat-type mean.

The raw group labels are cleaned by the shared preprocessor before target features are attached, so their keys match the `compact` model inputs.

In [56]:
def preprocess_partition(preprocessor, raw_frame, fit=False):
    features = raw_frame.drop(columns=['period', TARGET], errors='ignore')
    return preprocessor.fit_transform(features) if fit else preprocessor.transform(features)

# Fit the shared preprocessor on the early validation history only.
validation_preprocessor = RentalPreprocessor(datasets)
fit_base = preprocess_partition(validation_preprocessor, raw_fit, fit=True)
valid_base = preprocess_partition(validation_preprocessor, raw_valid)

# The nearest station is a geographic categorical feature; it uses only supplied MRT coordinates.
def nearest_mrt_name(base_frame):
    stations = datasets['mrt'][['NAME', 'LATITUDE', 'LONGITUDE']].dropna().copy()
    station_coords_km = stations[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96]
    row_coords_km = base_frame[['LATITUDE', 'LONGITUDE']].to_numpy() * [111.0, 110.96]
    _, nearest_indices = cKDTree(station_coords_km).query(row_coords_km, k=1)
    return pd.Series(
        stations['NAME'].astype(object).to_numpy()[nearest_indices],
        index=base_frame.index, name='nearest_mrt_name', dtype=object,
    )

def nearest_geojson_name(base_frame, filename, name_field, feature_name):
    """Return the name of the closest GeoJSON point for each HDB location."""
    geojson_path = project_root / 'data' / 'possible_additional_features' / filename
    with geojson_path.open() as file:
        geojson = json.load(file)
    points = pd.DataFrame([
        {
            'longitude': feature['geometry']['coordinates'][0],
            'latitude': feature['geometry']['coordinates'][1],
            feature_name: feature['properties'][name_field],
        }
        for feature in geojson['features']
        if feature.get('geometry', {}).get('type') == 'Point'
    ]).dropna(subset=['longitude', 'latitude', feature_name])
    point_coords_km = points[['longitude', 'latitude']].to_numpy() * [110.96, 111.0]
    row_coords_km = base_frame[['LONGITUDE', 'LATITUDE']].to_numpy() * [110.96, 111.0]
    _, nearest_indices = cKDTree(point_coords_km).query(row_coords_km, k=1)
    return pd.Series(
        points[feature_name].astype(object).to_numpy()[nearest_indices],
        index=base_frame.index, name=feature_name, dtype=object,
    )

# Use cleaned group values from the shared preprocessor, paired with their original period/target.
def labelled_history(raw_frame, base_frame):
    result = base_frame[GROUP_COLUMNS].copy().reset_index(drop=True)
    result['period'] = raw_frame['period'].to_numpy()
    result[TARGET] = raw_frame[TARGET].to_numpy()
    return result

def unlabelled_history(raw_frame, base_frame):
    result = base_frame[GROUP_COLUMNS].copy().reset_index(drop=True)
    result['period'] = raw_frame['period'].to_numpy()
    return result

fit_history = labelled_history(raw_fit, fit_base)
valid_history = labelled_history(raw_valid, valid_base)
validation_target_features = TargetFeatureTransformer(LAG_MONTHS)
fit_target_features = validation_target_features.fit_transform_train(fit_history)
valid_target_features = validation_target_features.transform(valid_history)

fit_model_base = fit_base[compact_features].reset_index(drop=True)
valid_model_base = valid_base[compact_features].reset_index(drop=True)
fit_nearest_mrt = nearest_mrt_name(fit_base).reset_index(drop=True)
valid_nearest_mrt = nearest_mrt_name(valid_base).reset_index(drop=True)
fit_nearest_attraction = nearest_geojson_name(
    fit_base, 'TouristAttractions.geojson', 'PAGETITLE', 'nearest_tourist_attraction_name'
).reset_index(drop=True)
valid_nearest_attraction = nearest_geojson_name(
    valid_base, 'TouristAttractions.geojson', 'PAGETITLE', 'nearest_tourist_attraction_name'
).reset_index(drop=True)
fit_nearest_monument = nearest_geojson_name(
    fit_base, 'MonumentsGEOJSON.geojson', 'NAME', 'nearest_monument_name'
).reset_index(drop=True)
valid_nearest_monument = nearest_geojson_name(
    valid_base, 'MonumentsGEOJSON.geojson', 'NAME', 'nearest_monument_name'
).reset_index(drop=True)
fit_model = pd.concat([
    fit_model_base, fit_target_features, fit_nearest_mrt, fit_nearest_attraction, fit_nearest_monument
], axis=1)
valid_model = pd.concat([
    valid_model_base, valid_target_features, valid_nearest_mrt, valid_nearest_attraction, valid_nearest_monument
], axis=1)
y_fit = raw_fit[TARGET].astype(float).reset_index(drop=True)
y_valid = raw_valid[TARGET].astype(float).reset_index(drop=True)

assert len(fit_model) == len(y_fit) and len(valid_model) == len(y_valid)
assert fit_model.columns.is_unique and valid_model.columns.is_unique
display(fit_model.head())


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


,TOWN,BLOCK,STREET,FLAT_TYPE,FLAT_MODEL,FLOOR_AREA_SQM,approval_date_num,approval_month,coe_quota_premium_mean,YEAR_COMPLETED,MAX_FLOOR,mrt_dist_nearest_km,mall_dist_nearest_km,monthly_target_lag,town_flat_type_target_lag,town_flat_type_target_mean,nearest_mrt_name,nearest_tourist_attraction_name,nearest_monument_name
0,punggol,604a,punggol road,4 room,model a,93.0,24253,1,45187.375,2010,16,0.555635,0.650683,NaN,NaN,NaN,punggol mrt/lrt station,The Japanese Cemetery Park,Church of the Nativity of the Blessed Virgin Mary
1,sengkang,290c,compassvale crescent,5 room,improved,110.0,24253,1,45187.375,2000,15,0.633253,0.615214,NaN,NaN,NaN,sengkang mrt/lrt station,The Japanese Cemetery Park,Church of the Nativity of the Blessed Virgin Mary
2,queenstown,5,ghim moh road,3 room,improved,65.0,24253,1,45187.375,1975,15,0.624787,0.603773,NaN,NaN,NaN,buona vista mrt station,"NUS University Museum, Singapore",Chinese High School Clock Tower
3,sengkang,436b,fernvale road,4 room,model a,96.0,24253,1,45187.375,2008,21,2.066014,0.182555,NaN,NaN,NaN,sengkang mrt/lrt station,The Japanese Cemetery Park,Church of the Nativity of the Blessed Virgin Mary
4,sembawang,467b,admiralty drive,4 room,premium apartment,102.0,24253,1,45187.375,1999,16,0.760948,0.729008,NaN,NaN,NaN,sembawang mrt station,Singapore Zoo: Attractions & Things to Do,Former Admiralty House


In [57]:
FEATURE_SETS = {
    'compact_plus_global_lag_mean_mrt_attraction_monument': compact_features + [
        'monthly_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name',
        'nearest_tourist_attraction_name', 'nearest_monument_name',
    ],
    'compact_plus_town_flat_type_lag_mean_mrt_attraction_monument': compact_features + [
        'town_flat_type_target_lag', 'town_flat_type_target_mean', 'nearest_mrt_name',
        'nearest_tourist_attraction_name', 'nearest_monument_name',
    ],
}

validation_results = []
for subset_name, columns in FEATURE_SETS.items():
    predictor = TabularPredictor(
        label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
        path=str(experiment_dir / 'ag_exp_9_models' / RUN_ID / f'validation_{subset_name}'),
    ).fit(
        train_data=fit_model[columns].assign(**{TARGET: y_fit}),
        tuning_data=valid_model[columns].assign(**{TARGET: y_valid}),
        hyperparameters=MODEL_HYPERPARAMETERS, presets=['medium_quality', 'ignore_text'],
        time_limit=TIME_LIMIT_SECONDS, verbosity=2,
    )
    predictions = predictor.predict(valid_model[columns]).astype(float).clip(lower=0)
    validation_results.append({
        'feature_set': subset_name, 'feature_count': len(columns),
        'RMSE': mean_squared_error(y_valid, predictions) ** 0.5,
        'MAE': mean_absolute_error(y_valid, predictions),
        'R2': r2_score(y_valid, predictions),
    })

validation_metrics = pd.DataFrame(validation_results).sort_values('RMSE').reset_index(drop=True)
display(validation_metrics)
print('Compare these results with the Experiment 7 compact RMSE of 467.230.')


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.31 GB / 16.00 GB (20.7%)
Disk Space Avail:   181.43 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_9_models/20260924_191005/validation_compact_plus_global_lag_mean_mrt_attraction_monument"
Train Data Rows:    117988
Train Data Columns: 18
Tuning Data Rows:    32012
Tuning Data Columns: 18
Label Column:    

,feature_set,feature_count,RMSE,MAE,R2
0,compact_plus_global_lag_mean_mrt_attraction_mo...,18,467.122316,340.293168,0.494477
1,compact_plus_town_flat_type_lag_mean_mrt_attra...,18,468.986498,342.418830,0.490434


Compare these results with the Experiment 7 compact RMSE of 467.230.


In [58]:
# After selecting the winning validation variant, refit its features and model on all labelled rows.
final_preprocessor = RentalPreprocessor(datasets)
full_base = preprocess_partition(final_preprocessor, raw_train, fit=True)
test_base = preprocess_partition(final_preprocessor, raw_test)
full_history = labelled_history(raw_train, full_base)
test_history = unlabelled_history(raw_test, test_base)
final_target_features = TargetFeatureTransformer(LAG_MONTHS)
full_target_features = final_target_features.fit_transform_train(full_history)
test_target_features = final_target_features.transform(test_history)
full_nearest_mrt = nearest_mrt_name(full_base).reset_index(drop=True)
test_nearest_mrt = nearest_mrt_name(test_base).reset_index(drop=True)
full_nearest_attraction = nearest_geojson_name(
    full_base, 'TouristAttractions.geojson', 'PAGETITLE', 'nearest_tourist_attraction_name'
).reset_index(drop=True)
test_nearest_attraction = nearest_geojson_name(
    test_base, 'TouristAttractions.geojson', 'PAGETITLE', 'nearest_tourist_attraction_name'
).reset_index(drop=True)
full_nearest_monument = nearest_geojson_name(
    full_base, 'MonumentsGEOJSON.geojson', 'NAME', 'nearest_monument_name'
).reset_index(drop=True)
test_nearest_monument = nearest_geojson_name(
    test_base, 'MonumentsGEOJSON.geojson', 'NAME', 'nearest_monument_name'
).reset_index(drop=True)
full_model = pd.concat([
    full_base[compact_features].reset_index(drop=True), full_target_features, full_nearest_mrt,
    full_nearest_attraction, full_nearest_monument,
], axis=1)
test_model = pd.concat([
    test_base[compact_features].reset_index(drop=True), test_target_features, test_nearest_mrt,
    test_nearest_attraction, test_nearest_monument,
], axis=1)
y_full = raw_train[TARGET].astype(float).reset_index(drop=True)

for subset_name, columns in FEATURE_SETS.items():
    predictor = TabularPredictor(
        label=TARGET, problem_type='regression', eval_metric='root_mean_squared_error',
        path=str(experiment_dir / 'ag_exp_9_models' / RUN_ID / f'final_{subset_name}'),
    ).fit(
        train_data=full_model[columns].assign(**{TARGET: y_full}),
        hyperparameters=MODEL_HYPERPARAMETERS, presets=['medium_quality', 'ignore_text'],
        time_limit=TIME_LIMIT_SECONDS, verbosity=2,
    )
    submission = pd.DataFrame({
        'Id': raw_test.index,
        'Predicted': predictor.predict(test_model[columns]).astype(float).clip(lower=0),
    })
    submission_path = experiment_dir / f'exp_9_{subset_name}_submission.csv'
    submission.to_csv(submission_path, index=False)
    print(f'Saved {subset_name} submission to {submission_path}')


[categorical_clean] HDB lookup | TOWN: 26 -> 26 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | BLOCK: 2735 -> 2735 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | STREET: 576 -> 576 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | POSTAL_CODE: 9180 -> 9180 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | SUBZONE: 156 -> 156 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces
[categorical_clean] HDB lookup | PLANNING_AREA: 31 -> 31 unique values | lowercase, trim, collapse whitespace, and replace hyphen/underscore/slash separators with spaces


Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.6.2
Python Version:     3.12.14
Operating System:   Darwin
Platform Machine:   arm64
Platform Version:   Darwin Kernel Version 25.6.0: Fri Jul 31 19:16:20 PDT 2026; root:xnu-12377.161.14~5/RELEASE_ARM64_T8142
CPU Count:          10
Pytorch Version:    2.10.0
CUDA Version:       CUDA is not available
GPU Count:          WARNING: Exception was raised when calculating GPU count (AssertionError)
Memory Avail:       3.27 GB / 16.00 GB (20.4%)
Disk Space Avail:   181.35 GB / 460.38 GB (39.4%)
Presets specified: ['medium_quality', 'ignore_text']
Beginning AutoGluon training ... Time limit = 600s
AutoGluon will save models to "/Users/metul/Desktop/KDDM-project/exploration/metul/ag_exp_9_models/20260924_191005/final_compact_plus_global_lag_mean_mrt_attraction_monument"
Train Data Rows:    150000
Train Data Columns: 18
Label Column:       MONTHLY_RENT
Problem Type:       regression
Preproces

Saved compact_plus_global_lag_mean_mrt_attraction_monument submission to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_9_compact_plus_global_lag_mean_mrt_attraction_monument_submission.csv


	Stage 1 Generators:
		Fitting AsTypeFeatureGenerator...
	Stage 2 Generators:
		Fitting FillNaFeatureGenerator...
	Stage 3 Generators:
		Fitting IdentityFeatureGenerator...
		Fitting CategoryFeatureGenerator...
			Fitting CategoryMemoryMinimizeFeatureGenerator...
	Stage 4 Generators:
		Fitting DropUniqueFeatureGenerator...
	Stage 5 Generators:
		Fitting DropDuplicatesFeatureGenerator...
	Types of features in original data (raw dtype, special dtypes):
		('float', [])        : 6 | ['FLOOR_AREA_SQM', 'coe_quota_premium_mean', 'mrt_dist_nearest_km', 'mall_dist_nearest_km', 'town_flat_type_target_lag', ...]
		('int', [])          : 4 | ['approval_date_num', 'approval_month', 'YEAR_COMPLETED', 'MAX_FLOOR']
		('object', [])       : 6 | ['TOWN', 'BLOCK', 'FLAT_TYPE', 'FLAT_MODEL', 'nearest_tourist_attraction_name', ...]
		('object', ['text']) : 2 | ['STREET', 'nearest_mrt_name']
	Types of features in processed data (raw dtype, special dtypes):
		('category', [])                   : 6 | ['TOWN'

Saved compact_plus_town_flat_type_lag_mean_mrt_attraction_monument submission to /Users/metul/Desktop/KDDM-project/exploration/metul/exp_9_compact_plus_town_flat_type_lag_mean_mrt_attraction_monument_submission.csv
